# MergeDNA results

Generates the figures used in the README from a trained checkpoint:

1. Training loss curve (MTR / latent MTR / AMTM)
2. Span-length histogram on a real DNA sequence
3. Span-length distribution by input content (repetitive vs random vs real)
4. Trained vs untrained model comparison

All figures are saved to `docs/` so the README can embed them. Run after `scripts/train_toy.py` has produced a checkpoint with `--checkpoint`.

In [ ]:
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from mergedna.data import encode_dna, read_sequences
from mergedna.model import MergeDNAConfig, MergeDNAModel

CHECKPOINT = ROOT / "checkpoints" / "ecoli.pt"
DOCS = ROOT / "docs"
DOCS.mkdir(exist_ok=True)

try:
    from torch.serialization import safe_globals
    with safe_globals([MergeDNAConfig]):
        payload = torch.load(CHECKPOINT, map_location="cpu")
except (ImportError, AttributeError):
    payload = torch.load(CHECKPOINT, map_location="cpu")

config = payload["config"]
model = MergeDNAModel(config)
model.load_state_dict(payload["model"])
model.eval()
history = payload.get("history", [])
print(f"loaded {CHECKPOINT.name}: {len(history)} history entries, seq_len={config.max_seq_len}, d_model={config.d_model}")

## 1. Training loss curves

Should all decrease and roughly plateau. AMTM starts much higher because it sums cross-entropy over masked bases.

In [ ]:
if not history:
    print("No history in checkpoint. Re-run train_toy.py with a recent commit.")
else:
    steps = [h["step"] for h in history]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(steps, [h["mtr"] for h in history], label="MTR")
    axes[0].plot(steps, [h["latent_mtr"] for h in history], label="latent MTR")
    axes[0].set_xlabel("step"); axes[0].set_ylabel("loss")
    axes[0].set_title("Reconstruction losses"); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(steps, [h["amtm"] for h in history], label="AMTM", color="C2")
    axes[1].set_xlabel("step"); axes[1].set_ylabel("loss")
    axes[1].set_title("AMTM (importance-weighted masked CE)"); axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(DOCS / "loss_curve.png", dpi=140, bbox_inches="tight")
    plt.show()
    print("saved", DOCS / "loss_curve.png")

## 2. Span-length histogram on a real DNA sequence

One token usually covers 1–4 bases. The shape of this distribution is the prototype's main qualitative result. A uniform tokenizer (`k`-mer, BPE) would give a single bar; MergeDNA spreads mass across span lengths.

In [ ]:
def span_distribution(seq: str, mdl: MergeDNAModel) -> Counter:
    pad = mdl.config.max_seq_len
    seq = seq[:pad]
    ids = encode_dna(seq, pad_to=pad).unsqueeze(0)
    with torch.no_grad():
        local = mdl.encode_local(ids)
    return Counter(len(g) for g in local.sources[0])

fasta = ROOT / "data" / "ecoli_k12_cds.fa"
if fasta.exists():
    sequences = read_sequences(fasta, min_length=config.max_seq_len)
    sample = sequences[0] if sequences else "ACGT" * (config.max_seq_len // 4)
    label = f"E. coli CDS (first sequence, {config.max_seq_len} bp)"
else:
    from mergedna.data import make_synthetic_dna
    sample = make_synthetic_dna(config.max_seq_len)
    label = f"synthetic ({config.max_seq_len} bp)"

dist = span_distribution(sample, model)
lengths, counts = zip(*sorted(dist.items()))
total_bases = sum(l * c for l, c in zip(lengths, counts))
total_tokens = sum(counts)
compression = total_bases / max(total_tokens, 1)
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(lengths, counts, color="C0")
ax.set_xlabel("span length (bases per merged token)")
ax.set_ylabel("count")
ax.set_title(f"Span-length distribution — {label}\ncompression {compression:.2f}×")
for xi, yi in zip(lengths, counts):
    ax.text(xi, yi, str(yi), ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.savefig(DOCS / "span_histogram.png", dpi=140, bbox_inches="tight")
plt.show()
print("saved", DOCS / "span_histogram.png")

## 3. Span distribution by input content

Three inputs of equal length: a single-base repeat, a dinucleotide repeat, and uniform-random DNA. The merge budget is fixed by `merge_ratio`, so total compression is identical — what matters is the *shape* of the span distribution.

**Caveat**: at small training budgets the differences are subtle and not always in the expected direction. The pair-selection step is non-differentiable, so `merge_key` only gets indirect signal; convincing content-aware merging needs longer training on more diverse data than a single bacterial genome.

In [ ]:
import random
random.seed(0)
L = config.max_seq_len
inputs = {
    "all-A": "A" * L,
    "CG repeat": ("CG" * (L // 2 + 1))[:L],
    "random": "".join(random.choices("ACGT", k=L)),
}
if fasta.exists() and sequences:
    inputs["E. coli CDS"] = sequences[0][:L]

max_span = 4
fig, ax = plt.subplots(figsize=(8, 4.5))
bar_width = 0.8 / len(inputs)
x = np.arange(1, max_span + 1)
for i, (name, seq) in enumerate(inputs.items()):
    dist = span_distribution(seq, model)
    heights = [dist.get(s, 0) for s in x]
    ax.bar(x + (i - len(inputs) / 2 + 0.5) * bar_width, heights, bar_width, label=name)
ax.set_xticks(x)
ax.set_xlabel("span length (bases)")
ax.set_ylabel("count")
ax.set_title("Span-length distribution by input content")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(DOCS / "span_by_content.png", dpi=140, bbox_inches="tight")
plt.show()
print("saved", DOCS / "span_by_content.png")

## 4. Trained vs untrained

Same architecture, but with random init instead of the trained weights. Differences here are evidence that training is shifting the merge decisions, not just that the architecture has a default behaviour.

In [ ]:
torch.manual_seed(0)
untrained = MergeDNAModel(config).eval()

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, mdl, title in [(axes[0], untrained, "Untrained"), (axes[1], model, "Trained")]:
    for name, seq in inputs.items():
        dist = span_distribution(seq, mdl)
        heights = [dist.get(s, 0) for s in x]
        ax.plot(x, heights, marker="o", label=name)
    ax.set_xticks(x); ax.set_xlabel("span length")
    ax.set_title(title); ax.grid(alpha=0.3); ax.legend(fontsize=8)
axes[0].set_ylabel("count")
plt.tight_layout()
plt.savefig(DOCS / "trained_vs_untrained.png", dpi=140, bbox_inches="tight")
plt.show()
print("saved", DOCS / "trained_vs_untrained.png")